In [ ]:
import pandas as pd
import numpy as np
import re

# Preproccing of the entries to achieve same format as provided by Jessa

In [ ]:
df = pd.read_feather("feather_files/mimiciv_cardiology_all_notes_with_icd.feather")
print(f"Length of df: {len(df)}")
print(f"Columns in df: {df.columns.tolist()}")
df.head()

In [ ]:
# Keep only rows where icd10_diag is not an empty list
df = df[df["icd10_diag"].apply(lambda x: len(x) > 0)].copy()
print(f"Remaining rows after dropping empty icd10_diag lists: {len(df)}")

df = df.drop(columns=["icd9_proc", "icd9_diag", "storetime", "note_type", "icd10_proc"])

In [ ]:
TARGET_ICD_CODES = {
    "diagnose_i480": ["I48.0"],
    "diagnose_i481": ["I48.1"],
    "diagnose_i482": ["I48.2"],
    "diagnose_i483": ["I48.3"],
    "diagnose_i484": ["I48.4"],
    "diagnose_i489": ["I48.9"],
    "diagnose_i501": ["I50.1"],
    "diagnose_i502": ["I50.2"],
    "diagnose_i503": ["I50.3"],
    "diagnose_i504": ["I50.4"],
    "diagnose_i508": ["I50.8"],
    "diagnose_i509": ["I50.9"],
    "diagnose_i69": ["I69"],
    "diagnose_n171819": ["N17", "N18", "N19"],
    "diagnose_z8673": ["Z86.73"],
    "diagnose_i10": ["I10"],
    "diagnose_i11": ["I11"],
    "diagnose_i12": ["I12"],
    "diagnose_i13": ["I13"],
    "diagnose_i15": ["I15"],
    "diagnose_e10": ["E10"],
    "diagnose_e11": ["E11"],
    "diagnose_e12": ["E12"],
    "diagnose_e13": ["E13"],
    "diagnose_e14": ["E14"],
}


def has_icd(code_list, targets):
    for code in code_list:
        for target in targets:
            if code.startswith(target):
                return 1
    return 0

In [ ]:
for col, icd_targets in TARGET_ICD_CODES.items():
    df[col] = df["icd10_diag"].apply(lambda x: has_icd(x, icd_targets))

In [ ]:
df.head()

In [ ]:
# Convert charttime/storetime to datetime if needed
df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")

# Sort by patient and time
df_sorted = df.sort_values(by=["subject_id", "charttime"], ascending=[True, True])

# Keep only the latest note per patient
df_latest = df_sorted.groupby("subject_id").tail(1).copy()

print(f"Original rows: {len(df)}, rows after taking latest note: {len(df_latest)}")

In [ ]:
df = df.drop(columns=["subject_id", "note_id", "note_seq", "charttime", "icd10_diag"])
print(df.columns.tolist())

In [ ]:
df.to_feather("mimiciv_cardiology_subset_prediction_25_labels_no_text_preproc.feather")

# Preprocessing of the "text" column

In [ ]:
# Show the first 5 notes' text
for i, text in enumerate(df["text"].head(5), 1):
    print(f"--- Note {i} ---")
    print(text[:1000])  # show first 1000 characters to avoid huge output
    print("\n")

First we remove the metadata.

In [ ]:
def remove_metadata(note: str) -> str:
    metadata_keys = [
        r"Name:",
        r"Unit No:",
        r"Admission Date:",
        r"Discharge Date:",
        r"Date of Birth:",
        r"Sex:",
        r"Service:",
        r"Allergies:",
        r"Attending:",
    ]

    pattern = re.compile(
        r"^(?:" + "|".join(metadata_keys) + r")\s*:?.*(?:\n[ \t]+.*)*",
        flags=re.MULTILINE | re.IGNORECASE,
    )

    cleaned_note = re.sub(pattern, "", note)

    cleaned_note = re.sub(r"\n\s*\n+", "\n\n", cleaned_note)

    return cleaned_note.strip()

In [ ]:
df["cleaned_text"] = df["text"].apply(remove_metadata)

In [ ]:
# Show the first 5 notes' text
for i, text in enumerate(df["cleaned_text"].head(5), 1):
    print(f"--- Note {i} ---")
    print(text[:1000])  # show first 1000 characters to avoid huge output
    print("\n")

Now we want to split the note into sections. 

In [ ]:
import pandas as pd
import re
import pprint

section_titles = [
    "major surgical or invasive procedure",
    "discharge condition",
    "followup instructions",
    "discharge medications",
    "pertinent results",
    "discharge instructions",
    "history of present illness",
    "social history",
    "family history",
    "discharge disposition",
    "chief complaint",
    "medications on admission",
    "discharge diagnosis",
    "brief hospital course",
    "past medical history",
    "facility",
    "physical exam",
    "activity status",
    "mental status",
    "level of consciousness",
]

section_pattern = re.compile(
    r"(?i)^(" + "|".join([re.escape(title) for title in section_titles]) + r")\s*[:\n]",
    re.MULTILINE,
)


def split_sections(text):
    sections = {}
    matches = list(section_pattern.finditer(text))
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_name = match.group(1).lower()
        sections[section_name] = text[start:end].strip()
    return sections


df["sections"] = df["cleaned_text"].apply(split_sections)
print(df["sections"].head())

In [ ]:
relevant_sections = [
    "discharge diagnosis",
    "brief hospital course",
    "major surgical or invasive procedure",
    "pertinent results",
    "history of present illness",
]

no_section_count = 0
no_section_indices = []

for idx, row in df.iterrows():
    note_sections = row["sections"]

    if not isinstance(note_sections, dict):
        no_section_count += 1
        no_section_indices.append(idx)
        continue

    combined_text = " ".join(
        (note_sections.get(sec, "") or "").strip() for sec in relevant_sections
    )

    if len(combined_text) == 0:
        no_section_count += 1
        no_section_indices.append(idx)

print(f"Notes with NONE of the relevant sections: {no_section_count}")
print(f"Percentage: {100 * no_section_count / len(df):.2f}%")

print(len(df))

In [ ]:
df.to_feather("mimiciv_cardiology_subset_prediction_25_labels_with_sections.feather")

# Training of custom Word2Vec model - for LAAT only
We first try to train it only on the remaining corpus.


In [ ]:
from gensim.models import Word2Vec
import re


## Tokenization ##
def simple_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens


tokenized_texts = [simple_tokenize(t) for t in df["text"].tolist()]
w2v_model = Word2Vec(
    sentences=tokenized_texts, vector_size=200, window=5, min_count=2, workers=8
)
w2v_model.save("mimiciv_w2v.model")


w2v_model.wv.save_word2vec_format("mimiciv_w2v.bin", binary=True)
print("Saved Word2Vec .bin file: mimiciv_w2v.bin")